In [ ]:
import json
import re
from collections import Counter

In [ ]:
def extract_cwe_statistics_primevul(input_file):
    cwe_counts = Counter()
    total_entries = 0
    missing_cwe_count = 0

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    entry = json.loads(line)
                    total_entries += 1
                    cwes = entry.get("cwe", [])
                    
                    if not cwes or (len(cwes) == 1 and str(cwes[0]).lower() == "none"):
                        continue
                    for cwe in cwes:
                        cwe_counts[cwe.strip()] += 1
                        
                except json.JSONDecodeError:
                    continue

        # Sort results: most frequent CWEs first
        sorted_cwes = sorted(cwe_counts.items(), key=lambda x: x[1], reverse=True)
        print("-" * 30)
        print(f"Total entries processed: {total_entries}")
        print(f"Unique CWE types found: {len(sorted_cwes)}")
        print("-" * 30)
        print(f"{'CWE ID':<15} | {'Frequency'}")
        print("-" * 30)
        
        for cwe, count in sorted_cwes[:10]:
            print(f"{cwe:<15} | {count}")
        print(f"...")

    except FileNotFoundError:
        print(f"Error: File '{input_file}' not found.")  

    return sorted_cwes  

In [ ]:
def extract_kb_cwes(input_file):
    cwe_distribution = Counter()
    total_entries = 0

    # Regex to find CWE- followed by one or more digits
    cwe_pattern = re.compile(r"CWE-\d+")

    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                
                try:
                    data = json.loads(line)
                    total_entries += 1
                    
                    # Target the metadata title
                    title = data.get("metadata", {}).get("title", "")
                    
                    if title:
                        match = cwe_pattern.search(title)
                        if match:
                            cwe_id = match.group()
                            # Increment the count for this specific CWE
                            cwe_distribution[cwe_id] += 1

                except json.JSONDecodeError:
                    continue

        # Sort the CWE keys numerically
        sorted_cwes = sorted(cwe_distribution.keys(), key=lambda x: int(x.split('-')[1]))

        print("-" * 40)
        print(f"Total knowledge items: {total_entries}")
        print(f"Unique CWE types found: {len(sorted_cwes)}")
        print("-" * 40)
        
        print(f"{'CWE ID':<15} | {'Frequency'}")
        print("-" * 30)
        
        # Iterate through the sorted list and pull counts from the distribution
        for cwe in sorted_cwes:
            count = cwe_distribution[cwe]
            print(f"{cwe:<15} | {count}")
        print("...")

    except FileNotFoundError:
        print(f"Error: Could not find {input_file}")
    
    return cwe_distribution, sorted_cwes

import json

def calculate_cwe_overlap(kb_cwe_distribution, kb_cwes, test_set_path):
    """
    kb_cwes: List of unique CWE strings returned by extract_kb_cwes()
    test_set_path: Path to the original_cpp_cwe_map.json file
    """
    try:
        with open(test_set_path, 'r', encoding='utf-8') as f:
            test_data = json.load(f)
        
        test_cwes = set(test_data.values())
        kb_cwes_set = set(kb_cwes)
        overlap = kb_cwes_set.intersection(test_cwes)
        # Number of KB entries that belong to overlapping CWEs
        kb_overlap_entries = sum(
            kb_cwe_distribution[cwe]
            for cwe in overlap
        )

        # Total KB entries with a recognized CWE
        total_kb_cwe_entries = sum(kb_cwe_distribution.values())

        kb_overlap_pct = (
            kb_overlap_entries / total_kb_cwe_entries * 100
            if total_kb_cwe_entries else 0
        )

        missing = test_cwes - kb_cwes_set
        coverage_pct = (len(overlap) / len(test_cwes)) * 100 if test_cwes else 0

        # Print results
        print("\n" + "="*40)
        print("OVERLAP ANALYSIS")
        print("="*40)
        print(f"Unique CWEs in Knowledge Base:  {len(kb_cwes_set)}")
        print(f"Unique CWEs in Test Set:        {len(test_cwes)}")
        print(f"CWEs found in both (Overlap):   {len(overlap)}")
        print(f"Test Set Coverage:              {coverage_pct:.2f}%")
        print(f"KB Entries Covered:            {kb_overlap_entries} / {total_kb_cwe_entries}")
        print(f"KB Coverage by Test CWEs:      {kb_overlap_pct:.2f}%")
        print("-" * 40)
        
        if overlap:
            print(f"Shared CWEs: {', '.join(sorted(overlap, key=lambda x: int(x.split('-')[1])))}")
        
        if missing:
            print("-" * 40)
            print(f"CWEs in Test Set NOT in KB ({len(missing)}):")
            print(f"{', '.join(sorted(missing, key=lambda x: int(x.split('-')[1])))}")
            
        return overlap

    except FileNotFoundError:
        print(f"Error: Could not find {test_set_path}")
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
def calculate_coverage(primevul_results, kb_results):
    # Convert to dictionaries/sets for comparison
    pv_dict = {cwe: count for cwe, count in primevul_results}
    pv_set = set(pv_dict.keys())
    kb_set = set(kb_results)

    # Intersection and Gaps
    common = pv_set.intersection(kb_set)
    missing_in_kb = pv_set - kb_set

    # Weighted Coverage Calculation
    total_pv_instances = sum(pv_dict.values())
    covered_pv_instances = sum(pv_dict[cwe] for cwe in common)
    weighted_cov = (covered_pv_instances / total_pv_instances * 100) if total_pv_instances > 0 else 0

    print("\n" + "="*50)
    print("FINAL COVERAGE ANALYSIS")
    print("="*50)
    print(f"Unique CWE types in PrimeVul:  {len(pv_set)}")
    print(f"Unique CWE types in KB:        {len(kb_set)}")

    print(f"Common CWE types:              {len(common)}")
    print("-" * 50)
    sorted_common = sorted(list(common), key=lambda x: int(x.split('-')[1]))
    for i in range(0, len(sorted_common), 4):
        row = sorted_common[i:i+4]
        print("  ".join(f"{cwe:<12}" for cwe in row))
    print("-" * 50)
    
    print(f"Type Coverage:     {len(common)/len(pv_set)*100:.2f}%")
    print(f"Weighted Coverage: {weighted_cov:.2f}% (Total PV samples covered)")
    print("-" * 50)

    if missing_in_kb:
        print("\nTop 10 Missing CWEs in KB (by PrimeVul frequency):")
        # Sort the missing IDs by their count in PrimeVul
        sorted_missing = sorted([(c, pv_dict[c]) for c in missing_in_kb], key=lambda x: x[1], reverse=True)
        for cwe, freq in sorted_missing[:10]:
            print(f" - {cwe:<10} | {freq} samples in PrimeVul")
    print("="*50)

In [ ]:
DATASET_PATH = "primevul_train_paired.jsonl"
primevul_train_cwes = extract_cwe_statistics_primevul(DATASET_PATH)

------------------------------
Total entries processed: 7578
Unique CWE types found: 112
------------------------------
CWE ID          | Frequency
------------------------------
CWE-119         | 1041
CWE-125         | 790
CWE-20          | 697
CWE-787         | 514
CWE-476         | 437
CWE-200         | 420
CWE-416         | 352
CWE-399         | 297
CWE-264         | 282
CWE-190         | 279
...


In [ ]:
KB_FILE = "../knowledge_db/cpp/code/export_json/docs.jsonl" 
kb_cwe_distribution, kb_cwes = extract_kb_cwes(KB_FILE)

----------------------------------------
Total knowledge items: 920
Unique CWE types found: 92
----------------------------------------
CWE ID          | Frequency
------------------------------
CWE-14          | 10
CWE-119         | 10
CWE-120         | 10
CWE-121         | 10
CWE-122         | 10
CWE-123         | 10
CWE-124         | 10
CWE-125         | 10
CWE-126         | 10
CWE-127         | 10
CWE-128         | 10
CWE-129         | 10
CWE-130         | 10
CWE-131         | 10
CWE-134         | 10
CWE-135         | 10
CWE-170         | 10
CWE-188         | 10
CWE-191         | 10
CWE-192         | 10
CWE-194         | 10
CWE-195         | 10
CWE-196         | 10
CWE-197         | 10
CWE-242         | 10
CWE-243         | 10
CWE-244         | 10
CWE-248         | 10
CWE-362         | 10
CWE-364         | 10
CWE-366         | 10
CWE-374         | 10
CWE-375         | 10
CWE-396         | 10
CWE-397         | 10
CWE-401         | 10
CWE-415         | 10
CWE-416         | 10
CWE-457

#### Overlap with Testset
*How many CWEs from UniVul are covered in the KB?*

In [ ]:
import json
from collections import Counter

# Load the JSON file
with open("original_cpp_cwe_map.json", "r", encoding="utf-8") as f:
    cve_to_cwe = json.load(f)

# Count CWE occurrences
cwe_counts = Counter(cve_to_cwe.values())

# Print statistics
total_cves = len(cve_to_cwe)
unique_cwes = len(cwe_counts)

print(f"Total CVEs: {total_cves}")
print(f"Unique CWEs: {unique_cwes}\n")

print("CWE distribution:")
for cwe, count in cwe_counts.most_common():
    pct = count / total_cves * 100
    print(f"{cwe:10} {count:3d} ({pct:5.1f}%)")

Total CVEs: 50
Unique CWEs: 23

CWE distribution:
CWE-20       8 ( 16.0%)
CWE-119      8 ( 16.0%)
CWE-190      3 (  6.0%)
CWE-264      3 (  6.0%)
CWE-125      3 (  6.0%)
CWE-189      2 (  4.0%)
CWE-399      2 (  4.0%)
CWE-200      2 (  4.0%)
CWE-347      2 (  4.0%)
CWE-704      2 (  4.0%)
CWE-835      2 (  4.0%)
CWE-787      2 (  4.0%)
CWE-532      1 (  2.0%)
CWE-310      1 (  2.0%)
CWE-17       1 (  2.0%)
CWE-269      1 (  2.0%)
CWE-354      1 (  2.0%)
CWE-416      1 (  2.0%)
CWE-184      1 (  2.0%)
CWE-209      1 (  2.0%)
CWE-362      1 (  2.0%)
CWE-59       1 (  2.0%)
CWE-78       1 (  2.0%)


In [ ]:
cwe_counts

Counter({'CWE-20': 8,
         'CWE-119': 8,
         'CWE-190': 3,
         'CWE-264': 3,
         'CWE-125': 3,
         'CWE-189': 2,
         'CWE-399': 2,
         'CWE-200': 2,
         'CWE-347': 2,
         'CWE-704': 2,
         'CWE-835': 2,
         'CWE-787': 2,
         'CWE-532': 1,
         'CWE-310': 1,
         'CWE-17': 1,
         'CWE-269': 1,
         'CWE-354': 1,
         'CWE-416': 1,
         'CWE-184': 1,
         'CWE-209': 1,
         'CWE-362': 1,
         'CWE-59': 1,
         'CWE-78': 1})

In [ ]:
overlap_list = calculate_cwe_overlap(kb_cwe_distribution, kb_cwes, 'original_cpp_cwe_map.json')


OVERLAP ANALYSIS
Unique CWEs in Knowledge Base:  92
Unique CWEs in Test Set:        23
CWEs found in both (Overlap):   6
Test Set Coverage:              26.09%
KB Entries Covered:            60 / 920
KB Coverage by Test CWEs:      6.52%
----------------------------------------
Shared CWEs: CWE-119, CWE-125, CWE-362, CWE-416, CWE-704, CWE-787
----------------------------------------
CWEs in Test Set NOT in KB (17):
CWE-17, CWE-20, CWE-59, CWE-78, CWE-184, CWE-189, CWE-190, CWE-200, CWE-209, CWE-264, CWE-269, CWE-310, CWE-347, CWE-354, CWE-399, CWE-532, CWE-835


#### Overlap with PrimeVul
*How many CWEs from PrimeVul are covered in the KB?*

In [ ]:
calculate_coverage(primevul_train_cwes, kb_cwes)


FINAL COVERAGE ANALYSIS
Unique CWE types in PrimeVul:  112
Unique CWE types in KB:        92
Common CWE types:              19
--------------------------------------------------
CWE-119       CWE-120       CWE-121       CWE-122     
CWE-125       CWE-129       CWE-131       CWE-134     
CWE-191       CWE-362       CWE-401       CWE-415     
CWE-416       CWE-476       CWE-704       CWE-763     
CWE-787       CWE-824       CWE-843     
--------------------------------------------------
Type Coverage:     16.96%
Weighted Coverage: 48.43% (Total PV samples covered)
--------------------------------------------------

Top 10 Missing CWEs in KB (by PrimeVul frequency):
 - CWE-20     | 697 samples in PrimeVul
 - CWE-200    | 420 samples in PrimeVul
 - CWE-399    | 297 samples in PrimeVul
 - CWE-264    | 282 samples in PrimeVul
 - CWE-190    | 279 samples in PrimeVul
 - CWE-703    | 256 samples in PrimeVul
 - CWE-189    | 245 samples in PrimeVul
 - CWE-772    | 143 samples in PrimeVul
 - CWE-

 19 common CWEs (like 119, 125, 416, and 476) are the "greatest hits" of memory safety and represent nearly half of the dataset

In [ ]:
from collections import Counter

filename = "primevul_sampled_univul_dist_manifest.txt"

cwe_counts = Counter()

with open(filename, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        # Skip header, separator, and empty lines
        if (not line or
            line.startswith("CWE_ID") or
            line.startswith("---")):
            continue

        # Split on "|" and strip whitespace
        parts = [p.strip() for p in line.split("|")]

        if len(parts) >= 4:
            cwe = parts[0]
            cwe_counts[cwe] += 1

total = sum(cwe_counts.values())

print(f"Total entries: {total}")
print(f"Unique CWEs: {len(cwe_counts)}\n")

print("CWE distribution:")
for cwe, count in cwe_counts.most_common():
    print(f"{cwe:10} {count:3d} ({count/total*100:5.1f}%)")

Total entries: 42
Unique CWEs: 18

CWE distribution:
CWE-119      8 ( 19.0%)
CWE-20       8 ( 19.0%)
CWE-125      3 (  7.1%)
CWE-190      3 (  7.1%)
CWE-189      2 (  4.8%)
CWE-200      2 (  4.8%)
CWE-399      2 (  4.8%)
CWE-704      2 (  4.8%)
CWE-787      2 (  4.8%)
CWE-835      2 (  4.8%)
CWE-17       1 (  2.4%)
CWE-264      1 (  2.4%)
CWE-269      1 (  2.4%)
CWE-310      1 (  2.4%)
CWE-354      1 (  2.4%)
CWE-362      1 (  2.4%)
CWE-416      1 (  2.4%)
CWE-59       1 (  2.4%)
